# Robust Device Architecture

This exercise deals with the robustness of embedded Linux devices against certain network-based threats.

## Preparation

For shell command calls, the Python module ``subprocess`` is used (https://docs.python.org/3/library/subprocess.html).

Run the following code block to provide the functions ``execute_vm_command()`` and ``execute_stm32_command()`` within this notebook.

In [ ]:
import subprocess

def execute_vm_command(cmd):
    run = subprocess.run(cmd, universal_newlines=True, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(run.stdout)
    
def execute_stm32_command(cmd):
    ssh = 'sshpass -p "root" ssh root@192.168.13.13 '
    run = subprocess.run((ssh + cmd), universal_newlines=True, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(run.stdout)

Of course, all commands given in this tutorial can also be executed directly on within a Linux console.

If you want to open a terminal window, execute the next code block.

In [ ]:
execute_vm_command('gnome-terminal')

---

### Serial Connection to the STM32MP1 Board

Within the following exercises, the network interface of the STM32MP1 device is stressed. In order to avoid latency issues with SSH during console printing, the serial interface has to be used for connecting to the board. The tool ``picocom`` can be used to connect to the board. Open a terminal and follow the instructions below.

In [ ]:
execute_vm_command('gnome-terminal')

**Step-by-Step Connection:**
- Connect the board (connector ``ST-Link``) to your PC with a micro USB cable
<br>
<img src="stm32mp1_serial_connection.png" width="400">
<br>
- Make sure the USB device is forwarded to the virtual machine
- Open a terminal and use ``picocom`` to connect to the board's UART interface:

```bash
sudo picocom /dev/ttyACM0 -b 115200
```

After boot-up, login to the board via the serial console with the default credentials ``root:root``.

**Notes:**
* If ``/dev/ttyACM0`` is not accessible, issue the command ``dmesg | grep tty`` to see if the device was registered in your VM. If not, there is a problem with the connection or the forwarding to the VM.
* If the device is named differently, e.g. ``ttyUSB0``, use that name instead of ``ttyACM0``.
* You can leave ``picocom`` by pressing ``CTRL+A`` followed by ``CTRL+X``.

---

## Exercise 10.1: Kernel Latency Measurement

This exercise focuses on kernel latency measurements to assess the impact of stress on real-time operations.

### Kernel Capabilities

In order to show the capabilities of the currently running kernel on the board, execute the following command via the serial console:

```bash
uname -a
```

**Task:**

Discuss the meaning of ``SMP`` and ``PREEMPT``.

**Meaning of ``SMP``:**

*TODO*

**Meaning of ``PREEMPT``:**

*TODO*

### Measuring Kernel Latency with ``cyclictest``

Start the tool ``cyclictest`` on the board on both cores with default priority to get a first impression:

```bash
cyclictest -S
```

This program measures the kernel latency in micro seconds, running a process on each of the both cores.

**Task:**

Take notes of the avg. and max. values after a few seconds and quit ``cyclictest`` with ``[CTRL + C]``.

**Measured Values:**

* AVG: *TODO*
* MAX: *TODO*

Now start ``cyclictest`` with real-time priority 80:

```bash
cyclictest -Sp80
```

**Tasks:**

* Again, take notes of the avg. and max. values after a few seconds and quit ``cyclictest`` with ``[CTRL + C]``.
* Compare the values to the previous results.

**Measured Values (RT Priority 80):**

* AVG: *TODO*
* MAX: *TODO*

**Differences to Measurement without RT Priotity:**

*TODO*

### Stress Generation Tool ``stress-ng``

``stress-ng`` can stress test a computer system in various selectable ways. It was designed to exercise various physical subsystems of a computer as well as a variety of operating system kernel interfaces. IT also has a wide range of CPU specific stress tests that perform floating point, integer, bit manipulation and control flow operations.

Use ``stress-ng`` to create synthetic 100% utilization on both cores (``--cpu 2``) including (hardware) crypto usage (``--af-alg 2``):

```bash
stress-ng --cpu 2 --af-alg 2 &
```

**Note:** You might need to press ``[RETURN]`` (a few times) to get your prompt back.

**Tasks:**

* Start ``cyclictest -S``, take notes and stop it.
* Start ``cyclictest -Sp80``, take notes and stop it.
* Compare avg. and max. values to the previous runs.

**Measurement Notes:**

*TODO*

**Comparison Insights:**

*TODO*

To stop the ``stress-ng`` background process, type ``killall stress-ng``.

---

## Exercise 10.2: Influences of Network Load

In this exercise, the impact of network traffic on kernel latency will be analyzed.

If not already connected, make sure the Ethernet connection between PC and STM32MP1 board is up and running.

### Measurement on the Board

Again, start the kernel latency measurement with the tool ``cyclictest`` and RT priority 80:

```bash
cyclictest -Sp80
```

### Execute Network Flooding Attack with ``hping3``

The tool ``hping3`` can be used to execute flooding attacks.

For this purpose, open a new terminal.

In [ ]:
execute_vm_command('gnome-terminal')

Within the terminal, execute a SYN flooding attack on the IP address of the STM32MP1 board:

```bash
sudo timeout 10 hping3 -S --flood 192.168.13.13
```

**Tasks:**

* Observe and note the changes in the terminal running ``cyclictest`` on the board.
* Discuss the consequences of the changes to kernel latency for the execution of a demanding real-time task.
* Discuss the differences that can be seen between the two cores.

**Changes in ``cyclictest`` Values while Flooding:**

*TODO*

**Consequences of Kernel Latency Changes:**

*TODO*

**Differences between Cores:**

*TODO*

Afterward, navigate to the terminal running ``cyclictest`` and stop the process by pressing ``[CTRL + C]``.

### Execute Network Scanning with ``nmap``

The tool ``hping3`` is meant to harm or disturb the target device. However, even less destructive tools like ``nmap`` for networking can have an impact on real-time tasks.

In the following experiment, the popular network scanning tool ``nmap`` is used to scan all open network ports on the STM32MP1 board with a SYN scan.

### Measurement on the Board

Again, start the kernel latency measurement with the tool ``cyclictest`` and RT priority 80:

```bash
cyclictest -Sp80
```

### Perform Network Scanning

Navigate to the terminal window previously running ``hping3``. 

Execute a SYN scan over ports 0 to 1023 of the board:

```bash
sudo nmap -sS 192.168.13.13 -p 0-1023
```

Run it a couple of times and watch the impact on the running ``cyclictest`` prompt.

**Tasks:**

* Note the max. values of the ``cyclictest`` output during scanning.
* Discuss the impact of the observed changes to the kernel latency and compare it to the previous measurement.
* Discuss the risks of scanning an industrial system with ``nmap``.

**Measurement Values:**

*TODO*

**Impact of Network Scanning on Real-Time Tasks:**

*TODO*

**Risks of Network Scanning in Industrial Systems:**

*TODO*

Afterward, navigate to the terminal running ``cyclictest`` and stop the process by pressing ``[CTRL + C]``.

### Final Discussion

How can engineers design embedded (real-time) systems that are robust against the shown attacks?

**Notes:**

*TODO*